# Implementasi Ant Colony Optimization pada Klasifikasi Tingkat Obesitas Menggunakan Random Forest

Notebook ini mereplikasi alur penelitian pada jurnal *"Implementation of Ant Colony Optimization in Obesity Level Classification Using Random Forest"* (JUTIF Vol. 6 No. 5, 2025):

1. Data Collection (dataset Obesity, 2111 data, 17 kolom, 7 kelas)
2. Pre-processing: Label Encoding → One-Hot Encoding → Duplicate Removal
3. Split data 80:20
4. Feature Selection dengan ACO (α=1, β=1, ρ=0.6, 16 semut, 5 iterasi)
5. Random Forest + 10-Fold Cross Validation (RF vs RF+ACO)
6. Evaluasi: classification report, confusion matrix, perbandingan metrik

> Hasil angka bisa sedikit berbeda dari jurnal karena ACO bersifat stokastik dan detail implementasi (fungsi heuristik, ukuran subset, hyperparameter RF) tidak dijelaskan lengkap di jurnal.

## 1. Import Library & Konfigurasi

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import mutual_info_classif
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support,
                             classification_report, confusion_matrix)

SEED = 42
np.random.seed(SEED)

# Parameter ACO (Tabel 3 jurnal)
ALPHA = 1.0        # bobot pheromone
BETA = 1.0         # bobot informasi heuristik
RHO = 0.6          # koefisien evaporasi
N_ANTS = 16        # jumlah semut
N_ITER = 5         # jumlah iterasi

DATA_PATH = "ObesityDataSet_raw_and_data_sinthetic.csv"

## 2. Data Collection

In [2]:
df = pd.read_csv(DATA_PATH)
print("Ukuran data :", df.shape)
df.head()

FileNotFoundError: [Errno 2] No such file or directory: 'ObesityDataSet_raw_and_data_sinthetic.csv'

In [ ]:
df.info()
print()
print(df['NObeyesdad'].value_counts())

## 3. Pre-processing

- **Label Encoding** untuk fitur biner: `family_history_with_overweight`, `FAVC`, `SMOKE`, `SCC` (0/1)
- **One-Hot Encoding** untuk fitur nominal: `Gender`, `CAEC`, `CALC`, `MTRANS` (dengan `drop_first=True` sehingga menghasilkan 23 fitur independen seperti pada Gambar 3 jurnal)
- **Duplicate Removal**: jurnal menemukan 24 data duplikat

In [ ]:
data = df.copy()

# Label encoding fitur biner
for col in ['family_history_with_overweight', 'FAVC', 'SMOKE', 'SCC']:
    data[col] = data[col].map({'no': 0, 'yes': 1})

# One-hot encoding fitur nominal
data = pd.get_dummies(data, columns=['Gender', 'CAEC', 'CALC', 'MTRANS'], drop_first=True, dtype=int)

# Label encoding target
classes = sorted(data['NObeyesdad'].unique())
label_map = {c: i for i, c in enumerate(classes)}
data['NObeyesdad_encoded'] = data['NObeyesdad'].map(label_map)
data = data.drop(columns='NObeyesdad')
print("Mapping kelas:", label_map)

# Hapus duplikat
print("Jumlah duplikat :", data.duplicated().sum())
data = data.drop_duplicates().reset_index(drop=True)
print("Ukuran setelah duplicate removal :", data.shape)

X = data.drop(columns='NObeyesdad_encoded')
y = data['NObeyesdad_encoded']
print("Jumlah fitur independen :", X.shape[1])
print(list(X.columns))

## 4. Split Data (80:20)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y)
print("Train:", X_train.shape, "| Test:", X_test.shape)

## 5. Feature Selection dengan Ant Colony Optimization

Representasi: setiap fitur adalah *node*, dan setiap semut membangun subset fitur.

- Probabilitas semut memilih fitur *j* (Persamaan 1): $P_j = \dfrac{\tau_j^{\alpha}\,\eta_j^{\beta}}{\sum_{l}\tau_l^{\alpha}\,\eta_l^{\beta}}$
- Informasi heuristik η: *mutual information* fitur terhadap target (dinormalisasi)
- Kualitas subset: akurasi Random Forest (3-fold CV pada data latih)
- Update pheromone (Persamaan 2): $\tau_j(t) = \rho\,\tau_j(t-1) + \sum_k \Delta\tau_j^k$, dengan $\Delta\tau_j^k$ = skor subset semut *k* jika fitur *j* dipilih
- Pheromone awal diinisialisasi acak (sesuai jurnal)
- Fitur terpilih: pheromone di atas rata-rata (Tabel 4)

In [ ]:
class ACOFeatureSelection:
    def __init__(self, n_ants=16, n_iter=5, alpha=1.0, beta=1.0, rho=0.6,
                 min_feats=5, max_feats=15, cv=3, rf_trees=50, seed=42):
        self.n_ants, self.n_iter = n_ants, n_iter
        self.alpha, self.beta, self.rho = alpha, beta, rho
        self.min_feats, self.max_feats = min_feats, max_feats
        self.cv, self.rf_trees = cv, rf_trees
        self.rng = np.random.default_rng(seed)
        self.seed = seed

    def _evaluate(self, X, y, idx):
        rf = RandomForestClassifier(n_estimators=self.rf_trees, random_state=self.seed, n_jobs=-1)
        skf = StratifiedKFold(self.cv, shuffle=True, random_state=self.seed)
        return cross_val_score(rf, X.iloc[:, idx], y, cv=skf, scoring='accuracy').mean()

    def fit(self, X, y):
        n = X.shape[1]
        # heuristik: mutual information (dinormalisasi, +epsilon agar tidak nol)
        mi = mutual_info_classif(X, y, random_state=self.seed)
        self.eta_ = (mi + 1e-6) / (mi + 1e-6).sum()
        # pheromone awal acak
        self.tau_ = self.rng.random(n)
        self.tau_ /= self.tau_.sum()

        self.best_score_, self.best_subset_ = -1, None
        self.history_ = []
        cache = {}

        for it in range(self.n_iter):
            prob = (self.tau_ ** self.alpha) * (self.eta_ ** self.beta)
            prob /= prob.sum()

            delta = np.zeros(n)
            iter_scores = []
            for _ in range(self.n_ants):
                k = self.rng.integers(self.min_feats, self.max_feats + 1)
                subset = tuple(sorted(self.rng.choice(n, size=k, replace=False, p=prob)))
                if subset not in cache:
                    cache[subset] = self._evaluate(X, y, list(subset))
                score = cache[subset]
                iter_scores.append(score)
                delta[list(subset)] += score
                if score > self.best_score_:
                    self.best_score_, self.best_subset_ = score, subset

            # update pheromone: evaporasi + deposit
            self.tau_ = self.rho * self.tau_ + delta
            self.tau_ /= self.tau_.sum()
            self.history_.append((it + 1, np.mean(iter_scores), np.max(iter_scores)))
            print(f"Iterasi {it+1}/{self.n_iter} | rata-rata skor: {np.mean(iter_scores):.4f} | terbaik: {np.max(iter_scores):.4f}")
        return self

aco = ACOFeatureSelection(n_ants=N_ANTS, n_iter=N_ITER, alpha=ALPHA, beta=BETA, rho=RHO, seed=SEED)
aco.fit(X_train, y_train)

### Pheromone tiap fitur (Gambar 4) dan fitur terpilih (Tabel 4)

In [ ]:
pher = pd.Series(aco.tau_, index=X.columns)
mean_pher = pher.mean()
print(f"Rata-rata pheromone: {mean_pher:.4f}")

plt.figure(figsize=(12, 4))
plt.bar(range(len(pher)), pher.values)
plt.axhline(mean_pher, color='red', ls='--', label=f'Rata-rata = {mean_pher:.4f}')
plt.xticks(range(len(pher)), pher.index, rotation=90)
plt.ylabel('Nilai pheromone'); plt.xlabel('Fitur')
plt.title('Pheromone value of features selected by ACO')
plt.legend(); plt.tight_layout(); plt.show()

selected = pher[pher > mean_pher].sort_values(ascending=False)
selected_features = list(selected.index)
print(f"\nJumlah fitur terpilih: {len(selected_features)} dari {X.shape[1]}")
pd.DataFrame({'Features': selected.index, 'Pheromone Value': selected.values.round(4)})

## 6. Cross-Validation 10-Fold (Tabel 5)

Perbandingan Random Forest dengan semua fitur (RF) vs fitur hasil ACO (RF+ACO) pada data latih.

In [ ]:
def make_rf():
    return RandomForestClassifier(n_estimators=100, random_state=SEED, n_jobs=-1)

skf10 = StratifiedKFold(n_splits=10, shuffle=True, random_state=SEED)
cv_rf = cross_val_score(make_rf(), X_train, y_train, cv=skf10, scoring='accuracy')
cv_aco = cross_val_score(make_rf(), X_train[selected_features], y_train, cv=skf10, scoring='accuracy')

tabel5 = pd.DataFrame({'RF': cv_rf, 'RF+ACO': cv_aco})
tabel5.index = [f'Fold {i}' for i in range(1, 11)]
tabel5.loc['Rata-rata'] = tabel5.mean()
(tabel5 * 100).round(2).astype(str) + '%'

## 7. Pembangunan Model & Evaluasi pada Data Uji (Tabel 6 & 7)

In [ ]:
target_names = classes

# --- RF dengan semua fitur ---
rf_all = make_rf().fit(X_train, y_train)
pred_all = rf_all.predict(X_test)

# --- RF dengan fitur hasil ACO ---
rf_aco = make_rf().fit(X_train[selected_features], y_train)
pred_aco = rf_aco.predict(X_test[selected_features])

print("=== Tabel 6: Random Forest (semua fitur) ===")
print(classification_report(y_test, pred_all, target_names=target_names, digits=2))
print(f"Overall Accuracy: {accuracy_score(y_test, pred_all)*100:.2f}%\n")

print("=== Tabel 7: Random Forest + ACO ===")
print(classification_report(y_test, pred_aco, target_names=target_names, digits=2))
print(f"Overall Accuracy: {accuracy_score(y_test, pred_aco)*100:.2f}%")

### Confusion Matrix

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
for ax, pred, title in zip(axes, [pred_all, pred_aco], ['RF (semua fitur)', 'RF + ACO']):
    cm = confusion_matrix(y_test, pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=target_names, yticklabels=target_names)
    ax.set_title(f'Confusion Matrix - {title}')
    ax.set_xlabel('Prediksi'); ax.set_ylabel('Aktual')
    ax.tick_params(axis='x', rotation=45)
plt.tight_layout(); plt.show()

### Perbandingan Metrik (Gambar 5)

In [ ]:
def mean_metrics(y_true, y_pred):
    p, r, f, _ = precision_recall_fscore_support(y_true, y_pred, average='macro', zero_division=0)
    return [p, r, f, accuracy_score(y_true, y_pred)]

labels = ['Precision (Mean)', 'Recall (Mean)', 'F1-Score (Mean)', 'Overall Accuracy']
m_rf, m_aco = mean_metrics(y_test, pred_all), mean_metrics(y_test, pred_aco)

x = np.arange(len(labels)); w = 0.35
fig, ax = plt.subplots(figsize=(9, 5))
b1 = ax.bar(x - w/2, np.array(m_rf)*100, w, label='RF')
b2 = ax.bar(x + w/2, np.array(m_aco)*100, w, label='RF+ACO')
ax.bar_label(b1, fmt='%.2f%%', fontsize=8); ax.bar_label(b2, fmt='%.2f%%', fontsize=8)
ax.set_xticks(x); ax.set_xticklabels(labels)
ax.set_ylim(90, 100); ax.set_ylabel('%')
ax.set_title('Model Performance Evaluation'); ax.legend()
plt.tight_layout(); plt.show()

pd.DataFrame({'RF': np.array(m_rf)*100, 'RF+ACO': np.array(m_aco)*100}, index=labels).round(2)

## 8. Perbandingan dengan Penelitian Sebelumnya (Tabel 8)

In [ ]:
hasil_ini = accuracy_score(y_test, pred_aco) * 100
tabel8 = pd.DataFrame({
    'Study': ['Dewi & Dwidasmara', 'Kivrak', 'Devi et al.', 'Alqahtani', 'Model ini (RF+ACO)', 'Jurnal (RF+ACO)'],
    'Algorithm': ['KNN', 'CNN', 'Decision Tree', 'MLP', 'RF+ACO', 'RF+ACO'],
    'Accuracy (%)': [78.98, 82.00, 94.00, 95.06, round(hasil_ini, 2), 96.17]})
tabel8

## 9. Kesimpulan

- ACO mereduksi fitur dari 23 menjadi subset yang lebih kecil (fitur dengan pheromone di atas rata-rata).
- Bandingkan akurasi RF vs RF+ACO pada Tabel 5–7 di atas dengan hasil jurnal (RF 94.49% → RF+ACO 96.17%).
- Catatan: pada jurnal, rata-rata akurasi 10-fold CV RF+ACO (93.93%) sebenarnya sedikit lebih rendah daripada RF (94.16%); peningkatan ke 96.17% terlihat pada evaluasi data uji 20%. Selisih ~1–2% bisa saja dipengaruhi pembagian data tertentu, sehingga perlu hati-hati dalam menarik kesimpulan.